In [2]:
import pandas as pd
import numpy as np

class SimpleDroneAnalyzer:
    """
    Simple analyzer to calculate total drone flight distance and total task completion time
    """
    
    def __init__(self, package_df=None, simulation_df=None):
        self.package_df = package_df
        self.simulation_df = simulation_df
        
    def load_data(self, package_file='package_simulation_records.csv', 
                  simulation_file='simulation_records.csv'):
        """Load simulation data from CSV files"""
        try:
            self.package_df = pd.read_csv(package_file)
            self.simulation_df = pd.read_csv(simulation_file)
            return True
        except Exception as e:
            print(f"Error loading data: {e}")
            return False
    
    def calculate_total_flight_distance(self):
        """Calculate total flight distance for all drones"""
        if self.simulation_df is None:
            return 0
        
        # Filter drone data
        drone_data = self.simulation_df[self.simulation_df['entity'] == 'drone'].copy()
        
        if drone_data.empty:
            return 0
        
        total_flight_distance = 0
        
        # Calculate distance for each drone
        for drone_id in drone_data['id'].unique():
            # Get single drone trajectory data, sorted by time
            single_drone = drone_data[drone_data['id'] == drone_id].sort_values('time')
            
            # Calculate movement distance for each step
            for i in range(1, len(single_drone)):
                prev_x = single_drone.iloc[i-1]['x']
                prev_y = single_drone.iloc[i-1]['y']
                curr_x = single_drone.iloc[i]['x']
                curr_y = single_drone.iloc[i]['y']
                
                # Calculate Euclidean distance
                distance = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
                total_flight_distance += distance
        
        return total_flight_distance
    
    def calculate_total_completion_time(self):
        """Calculate total completion time for all completed tasks"""
        if self.package_df is None:
            return 0
        
        total_completion_time = 0
        
        # Process each task
        for task_id in self.package_df['task_id'].unique():
            # Get all records for a single task, sorted by time
            task_data = self.package_df[self.package_df['task_id'] == task_id].sort_values('time')
            
            # Check if task is completed (has delivered or directly_delivered status)
            all_statuses = task_data['status'].unique()
            has_delivered_status = any(status in ['delivered', 'directly_delivered'] for status in all_statuses)
            
            if has_delivered_status:
                # Find start time: first time status changes from 'at_pickup_point' to something else
                start_time = None
                pickup_point_records = task_data[task_data['status'] == 'at_pickup_point']
                
                if not pickup_point_records.empty:
                    # Find the first time after 'at_pickup_point' when status changes
                    max_pickup_time = pickup_point_records['time'].max()
                    next_records = task_data[task_data['time'] > max_pickup_time]
                    if not next_records.empty:
                        start_time = next_records['time'].min()
                    else:
                        start_time = pickup_point_records['time'].min()
                else:
                    start_time = task_data['time'].min()
                
                # Find completion time: first time status becomes 'delivered' or 'directly_delivered'
                completion_records = task_data[task_data['status'].isin(['delivered', 'directly_delivered'])]
                
                if not completion_records.empty and start_time is not None:
                    end_time = completion_records['time'].min()
                    completion_time = end_time - start_time
                    total_completion_time += completion_time
        
        return total_completion_time
    
    def get_results(self):
        """Get both total flight distance and total completion time"""
        total_flight_distance = self.calculate_total_flight_distance()
        total_completion_time = self.calculate_total_completion_time()
        
        return total_flight_distance, total_completion_time

# Usage Functions
def analyze_from_dataframes(package_df, simulation_df):
    """Analyze data from existing DataFrames"""
    analyzer = SimpleDroneAnalyzer(package_df, simulation_df)
    return analyzer.get_results()

def analyze_from_files(package_file='package_simulation_records.csv', 
                      simulation_file='simulation_records.csv'):
    """Load and analyze data from CSV files"""
    analyzer = SimpleDroneAnalyzer()
    if analyzer.load_data(package_file, simulation_file):
        return analyzer.get_results()
    else:
        return 0, 0

# Main execution
if __name__ == "__main__":
    # Method 1: Analyze from CSV files
    total_flight_distance, total_completion_time = analyze_from_files()
    
    # Method 2: If you have DataFrames already loaded
    # total_flight_distance, total_completion_time = analyze_from_dataframes(package_df, simulation_df)
    
    print(f"Total Flight Distance: {total_flight_distance}")
    print(f"Total Completion Time: {total_completion_time}")

Total Flight Distance: 2816.098409787914
Total Completion Time: 842


In [7]:
import pandas as pd
import numpy as np
import os
import re

class BatchDroneAnalyzer:
    """
    批量分析多组无人机仿真数据
    """
    
    def __init__(self, folder_path='.'):
        self.folder_path = folder_path
        self.results = []
    
    def calculate_flight_distance(self, simulation_df):
        """计算总飞行距离"""
        if simulation_df is None or simulation_df.empty:
            return 0
        
        # 筛选无人机数据
        drone_data = simulation_df[simulation_df['entity'] == 'drone'].copy()
        
        if drone_data.empty:
            return 0
        
        total_flight_distance = 0
        
        # 计算每个无人机的距离
        for drone_id in drone_data['id'].unique():
            single_drone = drone_data[drone_data['id'] == drone_id].sort_values('time')
            
            # 计算每步的移动距离
            for i in range(1, len(single_drone)):
                prev_x = single_drone.iloc[i-1]['x']
                prev_y = single_drone.iloc[i-1]['y']
                curr_x = single_drone.iloc[i]['x']
                curr_y = single_drone.iloc[i]['y']
                
                distance = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
                total_flight_distance += distance
        
        return total_flight_distance
    
    def calculate_completion_time(self, package_df):
        """Calculate total completion time for all completed tasks"""
        if package_df is None:
            return 0
        
        total_completion_time = 0
        
        # Process each task
        for task_id in package_df['task_id'].unique():
            # Get all records for a single task, sorted by time
            task_data = package_df[package_df['task_id'] == task_id].sort_values('time')
            
            # Check if task is completed (has delivered or directly_delivered status)
            all_statuses = task_data['status'].unique()
            has_delivered_status = any(status in ['delivered', 'directly_delivered'] for status in all_statuses)
            
            if has_delivered_status:
                # Find start time: first time status changes from 'at_pickup_point' to something else
                start_time = None
                pickup_point_records = task_data[task_data['status'] == 'at_pickup_point']
                
                if not pickup_point_records.empty:
                    # Find the first time after 'at_pickup_point' when status changes
                    max_pickup_time = pickup_point_records['time'].max()
                    next_records = task_data[task_data['time'] > max_pickup_time]
                    if not next_records.empty:
                        start_time = next_records['time'].min()
                    else:
                        # If never left pickup point, use first pickup point time
                        start_time = pickup_point_records['time'].min()
                else:
                    # If no 'at_pickup_point' status found, use minimum time
                    start_time = task_data['time'].min()
                
                # Find completion time: first time status becomes 'delivered' or 'directly_delivered'
                completion_records = task_data[task_data['status'].isin(['delivered', 'directly_delivered'])]
                
                if not completion_records.empty and start_time is not None:
                    end_time = completion_records['time'].min()  # First time it became delivered
                    completion_time = end_time - start_time
                    total_completion_time += completion_time
        
        return total_completion_time
    
    def extract_drone_count(self, filename):
        """从文件名中提取无人机数量"""
        match = re.search(r'drones_(\d+)_cross', filename)
        return int(match.group(1)) if match else None
    
    def group_files(self):
        """将文件按无人机数量分组"""
        files = [f for f in os.listdir(self.folder_path) if f.endswith('.csv')]
        
        groups = {}
        for file in files:
            drone_count = self.extract_drone_count(file)
            if drone_count is not None:
                if drone_count not in groups:
                    groups[drone_count] = {'simulation': None, 'package': None, 'bus': None}
                
                if 'package_simulation_records' in file:
                    groups[drone_count]['package'] = file
                elif 'bus_simulation_records' in file:
                    groups[drone_count]['bus'] = file
                elif 'simulation_records' in file and 'package' not in file and 'bus' not in file:
                    groups[drone_count]['simulation'] = file
        
        return groups
    
    def analyze_single_group(self, drone_count, files):
        """分析单组文件"""
        try:
            # 读取文件
            package_df = None
            simulation_df = None
            
            if files['package']:
                package_path = os.path.join(self.folder_path, files['package'])
                package_df = pd.read_csv(package_path)
            
            if files['simulation']:
                simulation_path = os.path.join(self.folder_path, files['simulation'])
                simulation_df = pd.read_csv(simulation_path)
            
            # 计算指标
            total_flight_distance = self.calculate_flight_distance(simulation_df)
            total_completion_time = self.calculate_completion_time(package_df)
            
            return {
                'drone_count': drone_count,
                'total_flight_distance': total_flight_distance,
                'total_completion_time': total_completion_time,
                'cross_region': True,  # 所有文件都是cross版本
                'files_used': {
                    'package_file': files['package'],
                    'simulation_file': files['simulation'],
                    'bus_file': files['bus']
                }
            }
            
        except Exception as e:
            print(f"Error analyzing drone_count {drone_count}: {e}")
            return None
    
    def analyze_all_files(self):
        """分析所有文件并返回结果DataFrame"""
        file_groups = self.group_files()
        
        print(f"Found {len(file_groups)} drone count groups: {list(file_groups.keys())}")
        
        results = []
        
        for drone_count, files in sorted(file_groups.items()):
            print(f"\nAnalyzing {drone_count} drones...")
            print(f"  Package file: {files['package']}")
            print(f"  Simulation file: {files['simulation']}")
            print(f"  Bus file: {files['bus']}")
            
            result = self.analyze_single_group(drone_count, files)
            if result:
                results.append(result)
                print(f"  ✅ Total Flight Distance: {result['total_flight_distance']:.2f}")
                print(f"  ✅ Total Completion Time: {result['total_completion_time']:.2f}")
            else:
                print(f"  ❌ Failed to analyze")
        
        # 创建结果DataFrame
        if results:
            df = pd.DataFrame(results)
            # 重新排列列的顺序
            columns_order = ['drone_count', 'total_flight_distance', 'total_completion_time', 'cross_region']
            df = df[columns_order + [col for col in df.columns if col not in columns_order]]
            return df
        else:
            return pd.DataFrame()

def analyze_all_drone_files(folder_path='.'):
    """
    分析文件夹中所有无人机仿真文件
    
    Parameters:
    folder_path: 包含CSV文件的文件夹路径
    
    Returns:
    DataFrame: 包含每组无人机数量的总飞行距离和总完成时间
    """
    analyzer = BatchDroneAnalyzer(folder_path)
    return analyzer.analyze_all_files()

# 使用示例
if __name__ == "__main__":
    # 分析当前文件夹中的所有文件
    results_df = analyze_all_drone_files('wuhu')
    
    print("\n" + "="*60)
    print("📊 ALL SCENARIOS ANALYSIS RESULTS")
    print("="*60)
    print(results_df.to_string(index=False))
    
    # 保存结果到CSV
    results_df.to_csv('drone_analysis_results.csv', index=False)
    print(f"\n✅ Results saved to 'drone_analysis_results.csv'")

Found 0 drone count groups: []

📊 ALL SCENARIOS ANALYSIS RESULTS
Empty DataFrame
Columns: []
Index: []

✅ Results saved to 'drone_analysis_results.csv'
